# Shared analysis and plotting

Notebook-only plotting, diagnostic preprocessing, tables and exports for both research workflows. Run this notebook to define the helpers; it does not load data or train models. Numerical metrics, resampling algorithms, and training/data pipelines remain in the tested Python package.

## Graphs

In [ ]:
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, auc

from pumps_and_dumps.core.feature_type import FeatureType
from pumps_and_dumps.core.time_utils import NamedTimeDelta


# ---------------------------------------------------------------------------
# Feature distribution plots
# ---------------------------------------------------------------------------


def plot_feature_distributions(
    df_raw: pd.DataFrame,
    df_scaled: pd.DataFrame,
    feature_types: List[FeatureType],
    offset: NamedTimeDelta = NamedTimeDelta.ONE_DAY,
    n_pumps: int = 5,
    save_path: Optional[str] = None,
    random_state: int = 42,
) -> plt.Figure:
    """Plot feature distributions before/after cross-sectional standardisation."""
    selected_pump_hashes = np.random.default_rng(random_state).choice(
        sorted(df_raw["pump_hash"].unique()), n_pumps, replace=False
    )
    df_small = df_raw[df_raw["pump_hash"].isin(selected_pump_hashes)]
    df_scaled_small = df_scaled[df_scaled["pump_hash"].isin(selected_pump_hashes)]

    fig, axs = plt.subplots(len(feature_types), 2, figsize=(16, 8))
    for i, (row_axes, feature) in enumerate(zip(axs, feature_types)):
        ax1, ax2 = row_axes
        col_name = feature.col_name(offset=offset)

        sns.histplot(
            data=df_small,
            x=col_name,
            hue="pump_hash",
            ax=ax1,
            legend=False,
            alpha=0.05,
            bins=50,
            kde=True,
            stat="probability",
        )
        if i == 0:
            ax1.set_title(f"No standardisation: {col_name}")
        for ph in selected_pump_hashes:
            ax1.axvline(
                x=df_small.loc[
                    df_small["is_pumped"] & (df_small["pump_hash"] == ph), col_name
                ].iloc[0],
                color="red",
                linestyle="--",
            )

        sns.histplot(
            data=df_scaled_small,
            x=col_name,
            hue="pump_hash",
            ax=ax2,
            legend=False,
            alpha=0.05,
            bins=50,
            kde=True,
            stat="probability",
        )
        if i == 0:
            ax2.set_title(f"Cross-section standardisation: {col_name}")
        for ph in selected_pump_hashes:
            ax2.axvline(
                x=df_scaled_small.loc[
                    df_scaled_small["is_pumped"] & (df_scaled_small["pump_hash"] == ph),
                    col_name,
                ].iloc[0],
                color="red",
                linestyle="--",
            )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


# ---------------------------------------------------------------------------
# Top@k / Top@k% plots
# ---------------------------------------------------------------------------


def plot_topk_accuracy(
    df_topk: pd.DataFrame,
    cols: List[str],
    random_baseline: Optional[List[float]] = None,
    bins: Sequence[int] = (1, 2, 3, 5, 10, 20, 30),
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Plot Top@k accuracy curves with optional random baseline, anchored at (k=0, 0)."""
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot()
    origin = pd.DataFrame(0.0, index=[0], columns=cols)
    df_plot = pd.concat([origin, df_topk[cols].sort_index()])
    df_plot.plot(ax=ax, marker="o")
    if random_baseline is not None:
        plt.plot(
            [0, *bins],
            [0.0, *random_baseline],
            color="grey",
            marker="o",
            label="Random ranking baseline",
        )
    plt.legend()
    plt.ylabel("Top@k accuracy")
    plt.xlabel("k")
    plt.title("Top@k accuracy vs k")
    plt.grid()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


def plot_topk_percent_curves(
    df_topkp: pd.DataFrame,
    cols: List[str],
    auc_scores: Dict[str, float],
    max_k_percent: float = 0.20,
    save_path: Optional[str] = None,
    random_baseline: Optional[Sequence[float]] = None,
) -> plt.Figure:
    """Plot Top@k% accuracy curves with Top@k%AUC in legend over ``k% in [0, max_k_percent]``."""
    X = np.linspace(0, max_k_percent, 101)
    legends = [f"{c} - {auc_scores.get(c, 0):.3f}" for c in cols]

    fig, ax = plt.subplots(figsize=(10, 6))
    df_topkp[cols].plot(ax=ax, alpha=0.7)
    baseline_x = X if random_baseline is None else df_topkp.index
    baseline_y = X if random_baseline is None else random_baseline
    ax.plot(baseline_x, baseline_y, linestyle="--", color="grey", label="Random ranking baseline")
    ax.legend(legends + ["Random ranking baseline"])
    ax.set_xlabel("k%")
    ax.set_ylabel("Top@k% accuracy")
    ax.set_title("Top@k% accuracy vs k%")
    ax.set_xlim(0, max_k_percent)
    ax.grid()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


def plot_classification_metrics(
    df_metrics: pd.DataFrame,
    cols: List[str],
    title: str = "Classification metrics on TEST",
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Bar chart of classification metrics for selected models."""
    plot_cols = [c for c in cols if c in df_metrics.index]
    fig, ax = plt.subplots(figsize=(11, 5))
    df_metrics.loc[plot_cols].plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Score")
    ax.set_xlabel("Model")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


# ---------------------------------------------------------------------------
# Portfolio / PnL plots
# ---------------------------------------------------------------------------


def plot_equity_curves(
    df_curves: pd.DataFrame,
    cols: Optional[List[str]] = None,
    title: str = "Cumulative return on the test set for portfolio of size 5",
    save_path: Optional[str] = None,
    baseline_cols: Optional[List[str]] = None,
    random_col: Optional[str] = None,
    random_band: Optional[pd.DataFrame] = None,
) -> plt.Figure:
    """Cumulative equity curve plot with optional benchmark styling."""
    data = df_curves[cols] if cols else df_curves
    fig, ax = plt.subplots(figsize=(10, 6.2))
    marker = "o" if len(data.index) <= 100 else None
    data.cumsum().plot(ax=ax, marker=marker, x_compat=True)
    for line in ax.get_lines():
        if baseline_cols and line.get_label() in baseline_cols:
            line.set_color("black")
        if line.get_label() == random_col:
            line.set_color("dimgray")
            line.set_linestyle("--")
            line.set_linewidth(2)
    if random_band is not None:
        band = random_band.reindex(data.index).ffill().fillna(0)
        ax.fill_between(data.index, band["lower"], band["upper"], color="gray", alpha=0.18,
                        label="Random selection: pointwise 95% range")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)
    ax.set_title(title)
    ax.set_ylabel("Cumulative return")
    ax.axhline(0.0, color="grey", linewidth=0.8, alpha=0.6)
    ax.grid()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


def plot_pnl_sensitivity(
    df_pnl: pd.DataFrame,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """PnL sensitivity to intended order size with price impact."""
    if "cumulative_roe" in df_pnl.columns:
        metric_col = "cumulative_roe"
        metric_label = "Cumulative test-sample ROE"
        title = "Cumulative test-sample ROE by intended order size with price impact"
    elif "cumulative_roe_pct" in df_pnl.columns:
        metric_col = "cumulative_roe_pct"
        metric_label = "Cumulative test-sample ROE (%)"
        title = "Cumulative test-sample ROE by intended order size with price impact"
    elif "mean_roe" in df_pnl.columns:
        metric_col = "mean_roe"
        metric_label = "Mean ROE"
        title = "PnL sensitivity to intended order size with trade-level price impact"
    elif "mean_roe_pct" in df_pnl.columns:
        metric_col = "mean_roe_pct"
        metric_label = "Mean ROE (%)"
        title = "PnL sensitivity to intended order size with trade-level price impact"
    else:
        raise KeyError(
            "df_pnl must contain one of 'cumulative_roe', 'cumulative_roe_pct', 'mean_roe', or 'mean_roe_pct'"
        )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        df_pnl["quantity_usdt"],
        df_pnl[metric_col],
        color="tab:blue",
        linewidth=2,
        marker="o",
        label=metric_label,
    )
    ax.set_xscale("log")
    ax.set_xlabel("Intended order size (USDT)")
    ax.set_ylabel(metric_label)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


# ---------------------------------------------------------------------------
# Impact regression visualisation
# ---------------------------------------------------------------------------


def plot_impact_regression(
    model: "PriceImpactModel",  # noqa: F821
    samples: pd.DataFrame,
    currency_pair_name: str,
    pump_time: "datetime",  # noqa: F821
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Scatter + fitted sqrt impact curve for buy and sell sides (single beta, no intercept)."""
    b = model.beta
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    sides = [
        (axes[0], 1, "Buy", "#4C72B0"),
        (axes[1], -1, "Sell", "#DD8452"),
    ]

    # First pass: scatter points
    side_data = {}
    for ax, side, label, color in sides:
        side_df = samples[samples["side"] == side]
        side_data[side] = side_df

        ax.scatter(
            side_df["notional_usdt"],
            side_df["impact_bps"],
            s=14,
            alpha=0.25,
            color=color,
            edgecolors="none",
            rasterized=True,
        )
        ax.set_xlim(left=0)

    # Second pass: draw regression line spanning the full x-axis
    for ax, side, label, color in sides:
        side_df = side_data[side]
        if len(side_df) == 0:
            ax.set_title(f"{label} side  (n=0)", fontsize=12)
            continue

        x_max = ax.get_xlim()[1]
        curve_x = np.linspace(0, x_max, 500)
        curve_y = b * np.sqrt(curve_x)

        # R² calculation
        x_obs = side_df["notional_usdt"].values
        y_obs = side_df["impact_bps"].values
        y_pred = b * np.sqrt(x_obs)
        ss_res = np.sum((y_obs - y_pred) ** 2)
        ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

        ax.plot(
            curve_x,
            curve_y,
            color="black",
            linewidth=2.0,
            label=f"$\\beta$={b:.4f},  $R^2$={r2:.3f}",
            zorder=5,
        )

        ax.set_title(f"{label} side  (n={len(side_df):,})", fontsize=12)
        ax.set_xlabel("Net volume (USDT)", fontsize=11)
        ax.grid(alpha=0.25, linestyle="--")
        ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Impact (bps)", fontsize=11)
    fig.suptitle(
        f"Sqrt impact model: {currency_pair_name} before {pump_time:%Y-%m-%d %H:%M}",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


def plot_exit_impact_regression(
    model: "PriceImpactModel",  # noqa: F821
    samples: pd.DataFrame,
    currency_pair_name: str,
    pump_time: "datetime",  # noqa: F821
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Scatter + fitted sqrt impact curve for sell-only exit during manipulation window."""
    b = model.beta
    fig, ax = plt.subplots(figsize=(8, 5))

    sell_df = samples[samples["side"] == -1] if "side" in samples.columns else samples

    ax.scatter(
        sell_df["notional_usdt"],
        sell_df["impact_bps"],
        s=14,
        alpha=0.35,
        color="#C44E52",
        edgecolors="none",
        rasterized=True,
        label="Sell candles (5s)",
    )
    ax.set_xlim(left=0)

    if len(sell_df) > 0:
        x_max = ax.get_xlim()[1]
        curve_x = np.linspace(0, x_max, 500)
        curve_y = b * np.sqrt(curve_x)

        x_obs = sell_df["notional_usdt"].values
        y_obs = sell_df["impact_bps"].values
        y_pred = b * np.sqrt(x_obs)
        ss_res = np.sum((y_obs - y_pred) ** 2)
        ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

        ax.plot(
            curve_x,
            curve_y,
            color="black",
            linewidth=2.0,
            label=f"$\\beta$={b:.4f},  $R^2$={r2:.3f}",
            zorder=5,
        )

    ax.set_title(
        f"Exit impact (sell-only, 5s candles): {currency_pair_name}\n"
        f"10 min after pump at {pump_time:%Y-%m-%d %H:%M}  (n={len(sell_df):,})",
        fontsize=12,
    )
    ax.set_xlabel("Net sell volume (USDT)", fontsize=11)
    ax.set_ylabel("Absolute impact (bps)", fontsize=11)
    ax.grid(alpha=0.25, linestyle="--")
    ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


# ---------------------------------------------------------------------------
# Robustness plots
# ---------------------------------------------------------------------------


def _metric_label(col: str) -> str:
    """Map an internal metric column onto the paper's notation (``topk_percent@0.05`` -> ``Top@5%``)."""
    if col == "topk_percent_auc":
        return "Top@k%AUC"
    if col.startswith("topk_percent@"):
        return f"Top@{float(col.split('@', 1)[1]) * 100:g}%"
    if col.startswith("topk@"):
        return f"Top@{col.split('@', 1)[1]}"
    return col


def plot_robustness_distribution(
    df_robustness: pd.DataFrame,
    metric_cols: List[str],
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Histogram of Top@k%AUC and boxplot of per-bin Top@k% metrics."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    df_robustness["topk_percent_auc"].hist(ax=axes[0], bins=12, alpha=0.85)
    axes[0].axvline(
        df_robustness["topk_percent_auc"].mean(),
        color="red",
        linestyle="--",
        label="Mean",
    )
    axes[0].set_title("Distribution of Top@k%AUC")
    axes[0].set_xlabel("Top@k%AUC")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    df_robustness[metric_cols].rename(
        columns={c: _metric_label(c) for c in metric_cols}
    ).boxplot(ax=axes[1])
    axes[1].set_title("Distribution of Top@k% metrics")
    axes[1].set_ylabel("Score")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


def plot_feature_importance_distribution(
    feature_importances: pd.DataFrame,
    top_n: int = 12,
    n_cols: int = 3,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Grid of per-feature importance histograms across the bootstrap retraining runs.

    ``feature_importances`` is the long-format frame produced by
    :func:`run_cross_section_subset_robustness` with ``collect_feature_importances=True``
    (columns ``run_idx``, ``seed``, ``feature``, ``feature_importance``). The ``top_n``
    features with the highest median importance are shown, each as a histogram of its
    importance across runs with a dashed line at the median.
    """
    medians: pd.Series = (
        feature_importances.groupby("feature")["feature_importance"]
        .median()
        .sort_values(ascending=False)
    )
    top_features: List[str] = list(medians.head(top_n).index)

    n_rows: int = int(np.ceil(len(top_features) / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4 * n_cols, 2.6 * n_rows), squeeze=False
    )

    for idx, feature in enumerate(top_features):
        ax = axes[idx // n_cols][idx % n_cols]
        values: pd.Series = feature_importances.loc[
            feature_importances["feature"] == feature, "feature_importance"
        ]
        ax.hist(values, bins=10, alpha=0.85)
        ax.axvline(
            values.median(), color="red", linestyle="--", linewidth=1, label="Median"
        )
        ax.set_title(feature, fontsize=9)
        ax.set_xlabel("Importance")
        ax.set_ylabel("Count")

    for j in range(len(top_features), n_rows * n_cols):
        axes[j // n_cols][j % n_cols].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


# ---------------------------------------------------------------------------
# Bootstrap CI plots
# ---------------------------------------------------------------------------


def _pairwise_decisions(models: list[str], pairwise: pd.DataFrame) -> np.ndarray:
    """Validate the complete comparison family and orient each row's decision."""
    n_models = len(models)
    positions = {model: i for i, model in enumerate(models)}
    decisions = np.full((n_models, n_models), np.nan)
    for row in pairwise.itertuples(index=False):
        if row.model_a not in positions or row.model_b not in positions or row.model_a == row.model_b:
            raise ValueError("pairwise must compare distinct models present in summary")
        i, j = positions[row.model_a], positions[row.model_b]
        if np.isfinite(decisions[i, j]):
            raise ValueError("pairwise must contain exactly one row per unordered model pair")
        decision = 0
        if row.reject_h0:
            if row.better_model not in (row.model_a, row.model_b):
                raise ValueError("rejected comparisons must identify better_model")
            decision = 1 if row.better_model == row.model_a else -1
        decisions[i, j], decisions[j, i] = decision, -decision
    if np.isnan(decisions[~np.eye(n_models, dtype=bool)]).any():
        raise ValueError("pairwise must contain a comparison for every unordered model pair")

    return decisions


def _plot_model_intervals(ax, ordered: pd.DataFrame, identifiers: list[str], split: str, confidence: float) -> None:
    n_models = len(ordered)
    y = np.arange(n_models)
    # Draw intervals independently: a percentile CI need not contain its point estimate.
    ax.hlines(y, ordered["ci_lower"], ordered["ci_upper"], color="tab:blue", linewidth=2)
    ax.scatter(ordered["point_estimate"], y, color="tab:blue", zorder=3)
    ax.vlines(ordered["ci_lower"], y - 0.12, y + 0.12, color="tab:blue")
    ax.vlines(ordered["ci_upper"], y - 0.12, y + 0.12, color="tab:blue")
    ax.set_yticks(y, [f"{identifier}: {model}" for identifier, model in zip(identifiers, ordered.index)])
    ax.set_ylim(n_models - 0.5, -0.5)
    ax.set_xlabel(f"{split} Top@k%AUC (higher is better)")
    ax.set_title(f"Point estimates and {confidence:g}% pointwise bootstrap CIs")
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)


def _plot_pairwise_decisions(ax, decisions: np.ndarray, identifiers: list[str], alpha: float) -> None:
    n_models = len(identifiers)
    y = np.arange(n_models)
    colors = ["#d55e00", "#ededed", "#009e73"]
    cmap = ListedColormap(colors).with_extremes(bad="white")
    ax.imshow(np.ma.masked_invalid(decisions), cmap=cmap, vmin=-1.5, vmax=1.5)
    ax.set_xticks(y, identifiers)
    ax.set_yticks(y, identifiers)
    ax.set_xlabel("Column model (IDs match left panel)")
    ax.set_ylabel("Row model")
    ax.set_title(f"Paired two-sided tests; Holm-adjusted α = {alpha:g}")
    ax.set_xticks(np.arange(-0.5, n_models), minor=True)
    ax.set_yticks(np.arange(-0.5, n_models), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    labels = {-1: "−", 0: "·", 1: "+"}
    for (i, j), value in np.ndenumerate(decisions):
        ax.text(
            j,
            i,
            "—" if i == j else labels[value],
            ha="center",
            va="center",
            color="white" if abs(value) == 1 else "black",
        )
    ax.legend(
        handles=[
            Patch(facecolor=colors[2], label="+ Row significantly better"),
            Patch(facecolor=colors[0], label="− Row significantly worse"),
            Patch(facecolor=colors[1], label="· Inconclusive"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        frameon=False,
        fontsize=9,
    )


def plot_model_bootstrap_comparison(
    summary: pd.DataFrame,
    pairwise: pd.DataFrame,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Plot deterministic Top@k%AUC intervals and paired decisions.

    ``summary`` is indexed by model and contains ``point_estimate``, ``ci_lower``
    and ``ci_upper``. ``pairwise`` contains one row per unordered model pair,
    with ``model_a``, ``model_b``, ``reject_h0`` and ``better_model`` from
    :func:`compare_models_bootstrap_topk_percent_auc`. The optional ``alpha``
    column sets the confidence/significance labels (default: 0.05).

    Separate model intervals are pointwise; their overlap is never used to
    determine significance. Inconclusive comparisons do not establish equality.
    """
    if summary.empty or not summary.index.is_unique:
        raise ValueError("summary must contain at least one model and have a unique model index")
    ordered = summary.sort_values("point_estimate", ascending=False, kind="stable")
    metric_label = "Top@k%AUC"
    split = str(ordered["dataset_type"].iloc[0]).capitalize() if "dataset_type" in ordered else "Validation"
    models = list(ordered.index)
    alpha = float(ordered["alpha"].iloc[0]) if "alpha" in ordered.columns else 0.05
    confidence = 100 * (1 - alpha)
    n_models = len(models)

    decisions = _pairwise_decisions(models, pairwise)

    fig, (forest_ax, matrix_ax) = plt.subplots(
        1,
        2,
        figsize=(18, max(6, 0.48 * n_models + 2)),
        gridspec_kw={"width_ratios": [1.1, 1]},
        layout="constrained",
    )
    identifiers = [f"M{i + 1}" for i in range(n_models)]
    _plot_model_intervals(forest_ax, ordered, identifiers, split, confidence)
    _plot_pairwise_decisions(matrix_ax, decisions, identifiers, alpha)
    fig.suptitle(f"{split} model comparison: {metric_label}\n" "Paired bootstrap over pump events")
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
    return fig


def plot_bootstrap_ci(
    df_topk_ci: pd.DataFrame,
    df_topkp_ci: pd.DataFrame,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Error-bar plots for Top@k and Top@k% with bootstrap CIs."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    x = np.arange(len(df_topk_ci.index))
    y = df_topk_ci["point_estimate"].to_numpy()
    yerr = np.vstack([y - df_topk_ci["ci_lower"].to_numpy(), df_topk_ci["ci_upper"].to_numpy() - y])
    axes[0].errorbar(x, y, yerr=yerr, fmt="o-", capsize=4)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"Top@{k:g}" for k in df_topk_ci.index])
    axes[0].set_title("Top@k with 95% CI")
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Top@k")
    axes[0].grid(True)

    x2 = np.arange(len(df_topkp_ci.index))
    y2 = df_topkp_ci["point_estimate"].to_numpy()
    yerr2 = np.vstack(
        [
            y2 - df_topkp_ci["ci_lower"].to_numpy(),
            df_topkp_ci["ci_upper"].to_numpy() - y2,
        ]
    )
    axes[1].errorbar(x2, y2, yerr=yerr2, fmt="o-", capsize=4)
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels([f"Top@{k * 100:g}%" for k in df_topkp_ci.index])
    axes[1].set_title("Top@k% with 95% CI")
    axes[1].set_xlabel("k%")
    axes[1].set_ylabel("Top@k%")
    axes[1].grid(True)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    return fig


## Ranking report

In [ ]:
"""Interactive Top@k and Top@k% accuracy charts from computed metrics."""

from collections.abc import Mapping, Sequence
from pathlib import Path
from colorsys import hls_to_rgb
from hashlib import sha256

import pandas as pd
import plotly.graph_objects as go

from pumps_and_dumps.backtest.datasets.sample import DatasetType

_MODEL_COLORS = {
    "Logistic regression": "#1f77b4",
    "Random forest": "#ff7f0e",
    "CatBoost classifier": "#2ca02c",
    "CatBoost classifier + SMOTE": "#d62728",
    "CatBoost classifier + Top@k%AUC early stopping": "#9467bd",
    "CatBoost ranker": "#8c564b",
    "CatBoost + Top-5": "#e377c2",
    "uniform": "#20252b",
}


def _strategy_color(strategy: str) -> str:
    if strategy in _MODEL_COLORS:
        return _MODEL_COLORS[strategy]
    hue = int.from_bytes(sha256(strategy.encode()).digest()[:4], "big") / 2**32
    red, green, blue = hls_to_rgb(hue, 0.43, 0.65)
    return f"rgb({round(red * 255)}, {round(green * 255)}, {round(blue * 255)})"


def _strategy_label(strategy: str) -> str:
    return (
        strategy.replace(" + Tuned", " (tuned)")
        .replace("CatBoost classifier + Top@k%AUC early stopping", "CatBoost · Top@k%AUC")
        .replace(" + ", " · ")
    )


def _with_origin(points: pd.DataFrame) -> pd.DataFrame:
    ordered = points.sort_index().copy()
    if ordered.empty:
        raise ValueError("At least one ranking threshold is required.")
    if 0 not in ordered.index:
        ordered.loc[0] = 0.0
    return ordered.sort_index()


def _accuracy_trace(
    values: pd.Series,
    percent: bool,
    auc: float | None = None,
    baseline: bool = False,
) -> go.Scatter:
    threshold = "Top %{x:.1%}" if percent else "Top %{x:.0f}"
    auc_detail = f"<br>Top@k%AUC: {auc:.4f}" if auc is not None else ""
    name = str(values.name)
    return go.Scatter(
        x=values.index,
        y=values,
        name=name if baseline else _strategy_label(name),
        mode="lines",
        line={
            "color": _strategy_color("uniform" if baseline else name),
            "width": 2 if baseline else 2.5,
            "dash": "dash" if baseline else "solid",
        },
        hovertemplate=f"<b>{name}</b><br>{threshold}<br>Accuracy: %{{y:.2%}}{auc_detail}<extra></extra>",
    )


def plot_ranking_accuracy(
    points: pd.DataFrame,
    dataset_type: DatasetType = DatasetType.TEST,
    percent: bool = False,
    random_baseline: Sequence[float] | None = None,
    auc_scores: Mapping[str, float] | None = None,
    save_path: str | Path | None = None,
) -> go.Figure:
    """Plot hit rates indexed by K, or by a fraction of each event's candidates.

    Each column identifies a model. Optional baseline values follow the input
    index order and must use the same selection rule as the supplied metrics.
    AUC scores are displayed in hover details. HTML exports work offline.
    """
    plotted = points.copy()
    baseline_name = "Uniform random"
    if random_baseline is not None:
        plotted[baseline_name] = list(random_baseline)
    plotted = _with_origin(plotted)
    auc_scores = auc_scores or {}
    fig = go.Figure()
    for name in points.columns:
        fig.add_trace(_accuracy_trace(plotted[name], percent=percent, auc=auc_scores.get(name)))
    if random_baseline is not None:
        fig.add_trace(_accuracy_trace(plotted[baseline_name], percent=percent, baseline=True))

    metric = "Top@k%" if percent else "Top@k"
    fig.update_layout(
        template="plotly_white",
        height=680,
        autosize=True,
        margin={"l": 85, "r": 325, "t": 110, "b": 75},
        title={
            "text": f"{metric} accuracy · {dataset_type.name.lower()}",
            "x": 0.04,
            "subtitle": {
                "text": "Higher is better. Hover for accuracy; click a model to hide it, double-click to isolate it.",
                "font": {"size": 13, "color": "#49515c"},
            },
        },
        legend={
            "title": {"text": "Model"},
            "x": 1.02,
            "y": 1,
            "font": {"size": 11},
            "itemclick": "toggle",
            "itemdoubleclick": "toggleothers",
        },
        hovermode="closest",
        xaxis={
            "title": "Top fraction of candidate assets" if percent else "Number of top-ranked assets (K)",
            "tickformat": ".0%" if percent else "d",
            "rangemode": "tozero",
        },
        yaxis={"title": "Pumps with the true asset selected", "tickformat": ".0%", "range": [0, 1]},
    )
    if save_path is not None:
        destination = Path(save_path)
        if destination.suffix.lower() not in {".html", ".htm"}:
            raise ValueError("Interactive ranking charts must be saved as .html or .htm files.")
        destination.parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(
            destination,
            include_plotlyjs=True,
            full_html=True,
            config={"responsive": True, "scrollZoom": True, "displaylogo": False},
        )
    return fig


## Bootstrap report

In [ ]:
"""Readable, offline reports for paired bootstrap samples of whole test pumps."""

import json
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


if TYPE_CHECKING:
    from pumps_and_dumps.backtest.robustness.metric_bootstrap import TestMetricBootstrapResults


METRIC_LABELS = {
    "topk_percent_auc": "Top@k%AUC",
    "f1": "F1",
    "precision": "Precision",
    "recall": "Recall",
}


def _save_html(figure: go.Figure, save_path: str | Path | None) -> None:
    if save_path is None:
        return
    destination = Path(save_path)
    if destination.suffix.lower() not in {".html", ".htm"}:
        raise ValueError("Interactive bootstrap charts must be saved as .html or .htm files.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    figure.write_html(
        destination,
        include_plotlyjs=True,
        full_html=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )


def _score_range(values: np.ndarray) -> list[float]:
    lower, upper = float(values.min()), float(values.max())
    padding = max(0.02, (upper - lower) * 0.06)
    return [max(0.0, lower - padding), min(1.0, upper + padding)]


def _classification_note(top_k: int | None) -> str:
    if top_k == 1:
        return "With one true and one selected asset per pump, F1, precision and recall coincide."
    if top_k is not None:
        return f"Classification selects the top {top_k} assets in each pump."
    return "Classification uses the fixed selection rule from the evaluation."


def plot_metric_distributions(
    bootstrap_metrics: pd.DataFrame,
    save_path: str | Path | None = None,
) -> go.Figure:
    """Overlay the empirical distributions of four paired test-pump metrics.

    Each input row is one model evaluated on one bootstrap replicate; every
    replicate must contain every model. Ties contribute their full empirical
    mass at the observed score, without smoothing discrete classification
    metrics. The y-axis is the fraction of replicates at or below the score;
    distributions farther to the right have higher scores. One legend entry
    controls a model in all four panels. Exported HTML needs no network access.
    """
    if bootstrap_metrics.empty:
        raise ValueError("At least one bootstrap replicate is required.")
    models = bootstrap_metrics["model"].drop_duplicates().tolist()
    if (
        bootstrap_metrics.duplicated(["replicate", "model"]).any()
        or not (bootstrap_metrics.groupby("replicate").size() == len(models)).all()
    ):
        raise ValueError("Every bootstrap replicate must contain each model exactly once.")
    values = bootstrap_metrics[list(METRIC_LABELS)].to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError("Bootstrap metric values must be finite scores between zero and one.")

    figure = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=list(METRIC_LABELS.values()),
        vertical_spacing=0.22,
        horizontal_spacing=0.12,
    )
    for metric_index, (metric, label) in enumerate(METRIC_LABELS.items()):
        row, column = divmod(metric_index, 2)
        bounds = _score_range(bootstrap_metrics[metric].to_numpy())
        for model in models:
            scores = bootstrap_metrics.loc[bootstrap_metrics["model"] == model, metric].to_numpy()
            unique, counts = np.unique(scores, return_counts=True)
            # Include both sides of the first jump, even for constant metrics.
            x = np.r_[bounds[0], unique, bounds[1]]
            y = np.r_[0.0, np.cumsum(counts) / len(scores), 1.0]
            figure.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    name=_strategy_label(model),
                    legendgroup=model,
                    showlegend=metric_index == 0,
                    mode="lines",
                    line={"color": _strategy_color(model), "width": 2.5, "shape": "hv"},
                    hovertemplate=(
                        f"<b>{model}</b><br>{label}: %{{x:.3f}}"
                        "<br>Samples at or below score: %{y:.1%}<extra></extra>"
                    ),
                ),
                row=row + 1,
                col=column + 1,
            )
        figure.update_xaxes(
            title_text="Score (higher is better)", range=bounds, tickformat=".2f", row=row + 1, col=column + 1
        )
        figure.update_yaxes(
            title_text="Samples at or below score" if column == 0 else None,
            range=[0, 1.015],
            tickformat=".0%",
            row=row + 1,
            col=column + 1,
        )

    n_replicates = bootstrap_metrics["replicate"].nunique()
    note = _classification_note(bootstrap_metrics.attrs.get("classification_top_k"))
    figure.update_layout(
        template="plotly_white",
        autosize=True,
        height=920,
        margin={"l": 85, "r": 30, "t": 150, "b": 175},
        title={
            "text": f"Model metrics across {n_replicates:,} resampled test sets",
            "x": 0.04,
            "y": 0.98,
            "yanchor": "top",
            "font": {"size": 23},
            "subtitle": {
                "text": (
                    "The same whole-pump sample is used for every model. Curves farther right are better."
                    f"<br>{note}<br>Click a legend entry to hide a model in all panels; double-click to isolate it."
                ),
                "font": {"size": 12, "color": "#49515c"},
            },
        },
        legend={
            "orientation": "h",
            "x": 0,
            "y": -0.18,
            "yanchor": "top",
            "entrywidth": 0.32,
            "entrywidthmode": "fraction",
            "font": {"size": 11},
            "groupclick": "togglegroup",
            "itemclick": "toggle",
            "itemdoubleclick": "toggleothers",
        },
        hovermode="closest",
    )
    figure.update_xaxes(automargin=True)
    figure.update_yaxes(automargin=True)
    _save_html(figure, save_path)
    return figure


def plot_metric_intervals(
    metric_summary: pd.DataFrame,
    save_path: str | Path | None = None,
) -> go.Figure:
    """Compare original-test estimates and pointwise bootstrap intervals.

    A metric dropdown keeps model labels readable in a single full-width
    plot. The model order is fixed by original-test Top@k%AUC across metrics.
    Dots mark original-test estimates and horizontal segments mark percentile
    intervals; a point estimate need not lie inside its bootstrap interval.
    Interval overlap is not a paired hypothesis test. Use the separately
    exported, multiplicity-adjusted pairwise comparisons for such decisions.
    """
    if metric_summary.empty:
        raise ValueError("Metric summaries are required.")
    summary = metric_summary.set_index(["metric", "model"])
    if not summary.index.is_unique:
        raise ValueError("Each model and metric must have exactly one summary row.")
    models = (
        summary.loc["topk_percent_auc"].sort_values("point_estimate", ascending=False, kind="stable").index.tolist()
    )
    labels = [_strategy_label(model) for model in models]
    figure = go.Figure()
    buttons = []
    for metric_index, (metric, label) in enumerate(METRIC_LABELS.items()):
        selected = summary.loc[metric].loc[models]
        interval_x, interval_y = [], []
        for model, row in selected.iterrows():
            interval_x.extend([row["ci_lower"], row["ci_upper"], None])
            interval_y.extend([_strategy_label(model), _strategy_label(model), None])
        figure.add_trace(
            go.Scatter(
                x=interval_x,
                y=interval_y,
                mode="lines",
                line={"color": "#8995a5", "width": 4},
                hoverinfo="skip",
                visible=metric_index == 0,
                showlegend=False,
            )
        )
        figure.add_trace(
            go.Scatter(
                x=selected["point_estimate"],
                y=labels,
                mode="markers",
                marker={"color": [_strategy_color(model) for model in models], "size": 11},
                customdata=selected[["ci_lower", "ci_upper", "mean_bootstrap", "std_bootstrap"]].to_numpy(),
                hovertemplate=(
                    f"<b>%{{y}}</b><br>{label} on original test set: %{{x:.4f}}"
                    "<br>Percentile interval: [%{customdata[0]:.4f}, %{customdata[1]:.4f}]"
                    "<br>Bootstrap mean: %{customdata[2]:.4f}"
                    "<br>Bootstrap standard deviation: %{customdata[3]:.4f}<extra></extra>"
                ),
                visible=metric_index == 0,
                showlegend=False,
            )
        )
        buttons.append(
            {
                "label": label,
                "method": "update",
                "args": [
                    {"visible": [index // 2 == metric_index for index in range(2 * len(METRIC_LABELS))]},
                    {
                        "xaxis.title.text": f"{label} (higher is better)",
                        "xaxis.range": _score_range(selected[["point_estimate", "ci_lower", "ci_upper"]].to_numpy()),
                    },
                ],
            }
        )

    alpha = float(metric_summary["alpha"].iloc[0])
    note = _classification_note(
        int(metric_summary["classification_top_k"].iloc[0]) if "classification_top_k" in metric_summary else None
    )
    figure.update_layout(
        template="plotly_white",
        autosize=True,
        height=max(620, 45 * len(models) + 265),
        margin={"l": 260, "r": 35, "t": 175, "b": 80},
        title={
            "text": "Test-set estimates and bootstrap uncertainty",
            "x": 0.03,
            "y": 0.98,
            "yanchor": "top",
            "font": {"size": 23},
            "subtitle": {
                "text": (
                    f"Dots: original test set. Lines: {(1 - alpha):.0%} pointwise percentile intervals."
                    "<br>Use the paired comparison table to assess differences between models."
                    f"<br>{note}"
                ),
                "font": {"size": 12, "color": "#49515c"},
            },
        },
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 0, "y": 1.13, "xanchor": "left", "yanchor": "top"}],
        xaxis={
            "title": {"text": "Top@k%AUC (higher is better)"},
            "range": buttons[0]["args"][1]["xaxis.range"],
            "tickformat": ".2f",
            "automargin": True,
        },
        yaxis={"categoryorder": "array", "categoryarray": labels, "autorange": "reversed", "automargin": True},
    )
    _save_html(figure, save_path)
    return figure


def export_metric_bootstrap(results: "TestMetricBootstrapResults", output_dir: str | Path) -> dict[str, Path]:
    """Write all inputs needed to reconstruct each paired metric comparison.

    Sampled event IDs retain their replicate index and repeated draws. Per-event
    sufficient statistics and fitted-model scores make each resampled metric
    auditable without retraining. Metadata records evaluation settings, model
    and metric names, and the pairwise inference method. CSV tables preserve
    floating-point precision; the metadata and chart exports are independent.
    """
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    tables = {
        "bootstrap_metrics": results.bootstrap_metrics,
        "sampled_events": results.sampled_events.reset_index(),
        "event_metrics": results.event_metrics,
        "metric_summary": results.metric_summary,
        "pairwise": results.pairwise,
        "scored_predictions": pd.concat(
            [frame.assign(model=model) for model, frame in results.scored_datasets.items()], ignore_index=True
        ),
    }
    paths = {}
    for name, frame in tables.items():
        paths[name] = destination / f"{name}.csv"
        frame.to_csv(paths[name], index=False)
    metadata = {
        **results.bootstrap_metrics.attrs,
        "models": list(results.scored_datasets),
        "metrics": list(METRIC_LABELS),
        "pairwise_inference": dict(results.pairwise.attrs),
        "tables": {name: {"file": path.name, "rows": len(tables[name])} for name, path in paths.items()},
    }
    paths["metadata"] = destination / "metadata.json"
    paths["metadata"].write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return paths


## Impact example

In [ ]:
from dataclasses import dataclass
from datetime import timedelta
from typing import Optional

import pandas as pd
from pumps_and_dumps.backtest.portfolio.trades import HiveTradeSource

from pumps_and_dumps.backtest.portfolio.price_impact import (
    PriceImpactFitResult,
    fit_price_impact_model_from_klines_with_diagnostics,
    trades_to_klines,
)
from pumps_and_dumps.backtest.portfolio.indicative_price import IndicativePriceProvider
from pumps_and_dumps.backtest.datasets.sample import Dataset
from pumps_and_dumps.core.columns import (
    DATE,
    IS_BUYER_MAKER,
    PRICE,
    QUANTITY,
    SYMBOL,
    TRADE_TIME,
)
from pumps_and_dumps.core.currency_pair import CurrencyPair
from pumps_and_dumps.core.exchange import Exchange
from pumps_and_dumps.core.pump_event import PumpEvent
from pumps_and_dumps.core.time_utils import Bounds


@dataclass
class ImpactExample:
    pump: PumpEvent
    currency_pair: CurrencyPair
    fit_result: PriceImpactFitResult


@dataclass
class ExitImpactExample:
    pump: PumpEvent
    currency_pair: CurrencyPair
    fit_result: PriceImpactFitResult




def find_best_impact_example(
    dataset: Dataset,
    lookback_days: int = 14,
    buy_before: timedelta = timedelta(minutes=15),
    min_candles: int = 500,
    min_side_samples: int = 25,
    min_positive_impacts: int = 50,
) -> ImpactExample:
    """
    Search pump events in *dataset* for the best asset to illustrate the sqrt impact model.

    Loads only trades available before the simulated entry, resamples them
    into 5-minute candles, and fits using net volume as the order-flow proxy.
    """
    trade_source = HiveTradeSource()
    price_provider = IndicativePriceProvider()

    best: Optional[ImpactExample] = None
    best_score: Optional[tuple] = None

    for pump in dataset.get_pumps():
        cp = pump.currency_pair
        entry_time = pump.time - buy_before
        bounds = Bounds(
            start_inclusive=entry_time - timedelta(days=lookback_days),
            end_exclusive=entry_time,
        )
        trades = trade_source.load_trades(bounds, cp)
        if len(trades) < min_candles:
            continue

        klines = trades_to_klines(trades, freq="5min")
        if len(klines) < min_candles:
            continue

        try:
            q2u = price_provider.get_quote_to_usdt_indicative_price(cp.term, entry_time)
        except FileNotFoundError:
            q2u = 1.0

        result = fit_price_impact_model_from_klines_with_diagnostics(
            klines=klines,
            quote_to_usdt=q2u,
            sample_frequency="5min",
        )
        diag = result.diagnostics.set_index("side")
        buy_n = int(diag.loc["buy", "num_samples"]) if "buy" in diag.index else 0
        sell_n = int(diag.loc["sell", "num_samples"]) if "sell" in diag.index else 0
        pos = (
            int((result.samples["impact_bps"] > 0).sum())
            if not result.samples.empty
            else 0
        )

        if min(buy_n, sell_n) < min_side_samples or pos < min_positive_impacts:
            continue

        score = (min(buy_n, sell_n), pos)
        if best_score is None or score > best_score:
            best = ImpactExample(pump=pump, currency_pair=cp, fit_result=result)
            best_score = score

    if best is None:
        raise ValueError("No pump asset with usable kline-based impact samples found")
    return best


def find_best_exit_impact_example(
    dataset: Dataset,
    manipulation_window_minutes: int = 10,
    min_sell_samples: int = 20,
) -> ExitImpactExample:
    """
    Search pump events for the best asset to illustrate the exit impact model.

    Loads trades from the manipulation window (10 min after pump), resamples
    into 5-second candles, and fits using only sell-dominated candles.
    """
    trade_source = HiveTradeSource()
    price_provider = IndicativePriceProvider()

    best: Optional[ExitImpactExample] = None
    best_score: int = 0

    for pump in dataset.get_pumps():
        cp = pump.currency_pair
        bounds = Bounds(
            start_inclusive=pump.time,
            end_exclusive=pump.time + timedelta(minutes=manipulation_window_minutes),
        )
        trades = trade_source.load_trades(bounds, cp)
        if len(trades) < 50:
            continue

        klines = trades_to_klines(trades, freq="5s")
        if len(klines) < 10:
            continue

        try:
            q2u = price_provider.get_quote_to_usdt_indicative_price(cp.term, pump.time)
        except FileNotFoundError:
            q2u = 1.0

        result = fit_price_impact_model_from_klines_with_diagnostics(
            klines=klines,
            quote_to_usdt=q2u,
            sell_only=True,
            sample_frequency="5s",
        )

        n_samples = len(result.samples)
        if n_samples < min_sell_samples:
            continue

        if n_samples > best_score:
            best = ExitImpactExample(pump=pump, currency_pair=cp, fit_result=result)
            best_score = n_samples

    if best is None:
        raise ValueError("No pump asset with usable exit impact samples found")
    return best


## Diagnostics

In [ ]:
"""Dataset and impact-fit diagnostics used in the research notebook."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from pumps_and_dumps.backtest.experiment import Experiment
from pumps_and_dumps.backtest.pipelines.inspection import feature_importances
from pumps_and_dumps.core.feature_type import FeatureType
from pumps_and_dumps.features.schema import REGRESSOR_OFFSETS
from pumps_and_dumps.features.schema import regressor_columns


def plot_cross_section_sizes(df: pd.DataFrame, save_path: Path | None = None) -> plt.Figure:
    sizes = df.groupby("pump_hash").agg(
        pump_time=("pump_time", "first"), cross_section_size=("pump_time", "count")
    )
    ax = sizes.sort_values("pump_time").plot(
        x="pump_time", y="cross_section_size", title="Cross-section size over time"
    )
    ax.figure.tight_layout()
    if save_path is not None:
        ax.figure.savefig(save_path, dpi=300, bbox_inches="tight")
    return ax.figure


def describe_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return a diagnostic copy with clipped power-law values and feature statistics."""
    feature_cols = regressor_columns()
    powerlaw_cols = FeatureType.POWERLAW_ALPHA.col_names(offsets=REGRESSOR_OFFSETS)
    clipped = df.copy()
    clipped[powerlaw_cols] = clipped[powerlaw_cols].clip(1, 2)
    print(f"Dataset shape: {clipped.shape}")
    print(f"NaN rows remaining: {clipped[feature_cols].isna().any(axis=1).sum()}")
    return clipped, clipped[feature_cols].describe().T[["mean", "std", "min", "max"]]


def plot_feature_importances(
    experiment: Experiment, save_path: Path | None = None, top_n: int = 20
) -> plt.Figure:
    importance = feature_importances(experiment.get_model())
    ax = (
        importance.sort_values()
        .tail(top_n)
        .rename_axis("feature")
        .plot.barh(figsize=(10, 6))
    )
    ax.set_title(experiment.get_experiment_name())
    ax.figure.tight_layout()
    if save_path is not None:
        ax.figure.savefig(save_path)
    return ax.figure


def impact_fit_metrics(samples: pd.DataFrame, beta: float) -> pd.Series:
    """Compute the same fit diagnostics for entry and exit square-root models."""
    observed = samples["impact_bps"].to_numpy()
    predicted = beta * np.sqrt(samples["notional_usdt"].to_numpy())
    residuals = observed - predicted
    ss_total = np.sum((observed - np.mean(observed)) ** 2)
    return pd.Series(
        {
            "n": len(samples),
            "R²": 1 - np.sum(residuals**2) / ss_total if ss_total > 0 else 0.0,
            "MAE (bps)": np.mean(np.abs(residuals)),
            "RMSE (bps)": np.sqrt(np.mean(residuals**2)),
        }
    )


def show_impact_example(
    example: ImpactExample | ExitImpactExample, save_path: Path | None = None
) -> plt.Figure:
    """Plot an illustrative fit and report per-side errors; never used for model selection."""
    is_exit = isinstance(example, ExitImpactExample)
    samples = example.fit_result.samples.copy()
    if not is_exit:
        samples = samples.loc[samples["notional_usdt"] > 0]
    plot = (
        plot_exit_impact_regression if is_exit else plot_impact_regression
    )
    figure = plot(
        model=example.fit_result.model,
        samples=samples,
        currency_pair_name=example.currency_pair.name,
        pump_time=example.pump.time,
        save_path=str(save_path) if save_path is not None else None,
    )
    groups = (
        {"Exit (sell-only, 5s candles)": samples}
        if is_exit
        else {
            label: samples.loc[samples["side"] == side]
            for label, side in (("Buy", 1), ("Sell", -1))
        }
    )
    metrics = {
        label: impact_fit_metrics(group, example.fit_result.model.beta)
        for label, group in groups.items()
        if not group.empty
    }
    display(pd.DataFrame(metrics).T)
    print(f"β = {example.fit_result.model.beta:.6f}")
    print(f"Asset: {example.currency_pair.name}, Pump: {example.pump.time}")
    return figure


## Reports

In [ ]:
"""Assemble notebook tables and exports without changing evaluation or model selection."""

from collections.abc import Sequence
from pathlib import Path
from shutil import copyfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from pumps_and_dumps.backtest import evaluation
from pumps_and_dumps.backtest.experiment import Experiment
from pumps_and_dumps.backtest.datasets.sample import DatasetType
from pumps_and_dumps.backtest.robustness.validation_bootstrap import (
    ValidationBootstrapResults,
    comparisons_for_model,
)


def show_figure(figure: plt.Figure, filename: str, image_dir: Path, paper_dir: Path | None = None) -> None:
    """Display the saved Matplotlib image and copy those exact bytes to the paper."""
    image_dir.mkdir(parents=True, exist_ok=True)
    path = image_dir / filename
    try:
        figure.savefig(path, dpi=150, bbox_inches="tight", facecolor="white", transparent=False)
        if paper_dir is not None:
            paper_dir.mkdir(parents=True, exist_ok=True)
            copyfile(path, paper_dir / filename)
        display(Image(filename=str(path)))
    finally:
        plt.close(figure)  # Prevent a second automatic inline rendering.


def report_model_metrics(
    experiments: list[Experiment], dataset_type: DatasetType, output_dir: Path, paper_dir: Path
) -> pd.DataFrame:
    """Use the same three paper plots and compact metric table for either split."""
    split = dataset_type.name.lower()
    cols = [experiment.get_experiment_name() for experiment in experiments]
    topk = evaluation.evaluate_experiments_topk(experiments, dataset_type=dataset_type)
    topkp = evaluation.evaluate_experiments_topk_percent(experiments, dataset_type=dataset_type)
    curves, auc_scores = evaluation.evaluate_experiments_topk_percent_curves(
        experiments, bins=np.arange(0, 0.205, 0.005), dataset_type=dataset_type
    )
    classification = evaluation.evaluate_experiments_classification(experiments, dataset_type=dataset_type)
    dataset = experiments[0].get_sample().get_dataset(dataset_type)
    event_sizes = dataset.all_data().groupby("pump_hash").size().to_numpy()
    random_topkp = [(np.ceil(fraction * event_sizes) / event_sizes).mean() for fraction in curves.index]
    for filename, figure in (
        (
            f"topk_accuracy_{split}.png",
            plot_topk_accuracy(topk, cols, random_baseline=evaluation.random_topk_baseline(dataset)),
        ),
        (
            f"topkp_auc_{split}.png",
            plot_topk_percent_curves(curves, cols, auc_scores, random_baseline=random_topkp),
        ),
        (
            f"classification_metrics_{split}.png",
            plot_classification_metrics(classification, cols, title=f"Classification metrics on {split}"),
        ),
    ):
        show_figure(figure, filename, image_dir=output_dir / "images", paper_dir=paper_dir)
    table = validation_metrics_table(pd.Series(auc_scores), topk, topkp, classification)
    tables_dir = output_dir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)
    for name, values in {
        "topk": topk,
        "topkp": topkp,
        "curves": curves,
        "classification": classification,
        "metrics": table,
    }.items():
        values.to_csv(tables_dir / f"{split}_{name}.csv")
    display(table.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold;"))
    return table


def validation_metrics_table(
    auc_ranking: pd.Series,
    topk: pd.DataFrame,
    topk_percent: pd.DataFrame,
    classification: pd.DataFrame,
    auc_label: str = "Top@k%AUC",
) -> pd.DataFrame:
    """Align already computed ranking and classification metrics by model name."""
    table = auc_ranking.rename(auc_label).to_frame()
    for k in (1, 5, 10):
        table[f"Top@{k}"] = topk.T[k]
    for fraction, label in ((0.01, "Top@1%"), (0.05, "Top@5%"), (0.10, "Top@10%")):
        table[label] = topk_percent.T[fraction]
    for metric in (
        "PR-AUC",
        "F1 (Top@1 per cross-section)",
        "Balanced Accuracy (Top@1 per cross-section)",
    ):
        table[metric] = classification[metric]
    return table.rename(
        columns={
            "F1 (Top@1 per cross-section)": "F1",
            "Balanced Accuracy (Top@1 per cross-section)": "Balanced accuracy",
        }
    )


def report_bootstrap(
    results: ValidationBootstrapResults,
    notebook_dir: Path,
    n_bootstrap: int,
    dataset_type: DatasetType = DatasetType.VALIDATION,
    output_dir: Path | None = None,
) -> None:
    """Export the full comparison family and display the AUC summary without rescoring."""
    split = dataset_type.name.lower()
    if not results.auc_summary["dataset_type"].eq(dataset_type.name).all():
        raise ValueError("Bootstrap results must match the requested report split")
    metric_label = "Top@k%AUC"
    folder = "significance" if dataset_type is DatasetType.VALIDATION else f"significance_{split}"
    output = output_dir if output_dir is not None else notebook_dir / "analysis_outputs" / folder
    output.mkdir(parents=True, exist_ok=True)
    for filename, table in {
        f"{split}_topk_percent_auc_ci.csv": results.auc_summary,
        f"{split}_topk_ci.csv": results.topk_ci,
        f"{split}_topk_percent_ci.csv": results.topk_percent_ci,
    }.items():
        table.to_csv(output / filename)
    results.pairwise.to_csv(output / f"{split}_topk_percent_auc_pairwise.csv", index=False)
    print(
        f"{len(results.scored_datasets)} models; {len(results.pairwise)} paired comparisons; "
        f"{n_bootstrap:,} shared event resamples."
    )
    columns = [
        "point_estimate",
        "mean_bootstrap",
        "std_bootstrap",
        "ci_lower",
        "ci_upper",
        "n_cross_sections",
    ]
    display(
        results.auc_summary[columns]
        .style.format(precision=4)
        .set_caption(f"{split.capitalize()} {metric_label}: pointwise bootstrap intervals")
    )


def report_validation_bootstrap(
    results: ValidationBootstrapResults,
    notebook_dir: Path,
    n_bootstrap: int,
    output_dir: Path | None = None,
) -> None:
    """Preserve the validation report API and its existing output directory."""
    report_bootstrap(results, notebook_dir, n_bootstrap=n_bootstrap, output_dir=output_dir)


def report_chosen_comparisons(
    pairwise: pd.DataFrame,
    model_name: str,
    notebook_dir: Path,
    alpha: float = 0.05,
    output_dir: Path | None = None,
) -> pd.DataFrame:
    """Report one view of the existing Holm-adjusted comparisons, without retesting."""
    metric_label = "Top@k%AUC"
    chosen = comparisons_for_model(pairwise, model_name)
    for column in ("metric", "dataset_type"):
        if column in pairwise:
            chosen[column] = pairwise[column].iloc[0]
    output = output_dir if output_dir is not None else notebook_dir / "analysis_outputs" / "significance"
    output.mkdir(parents=True, exist_ok=True)
    chosen.to_csv(output / "validation_chosen_model_comparisons.csv")
    display(
        chosen.style.format(precision=4).set_caption(
            f"{metric_label}: chosen model minus comparator; " "decisions use Holm-adjusted p-values"
        )
    )
    wins = chosen.index[chosen["conclusion"] == "Chosen model better"].tolist()
    print(f"Chosen model: {model_name}")
    if wins:
        print(f"Validation evidence of higher {metric_label} (Holm-adjusted p < {alpha:g}) than: " + "; ".join(wins))
    else:
        print("No statistically significant superiority established for the chosen model on validation.")
    print(
        "These comparisons condition on fitted models. Tuning used a separate partition in this run; "
        "earlier research inspected these periods, so this is a retrospective comparison."
    )
    return chosen


def portfolio_benchmark(
    experiment: Experiment, portfolio_sizes: Sequence[int] = (1, 2, 5, 10, 20, 30)
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare fixed-size TEST portfolios with the time-aligned BTC benchmark."""
    curves = {
        k: evaluation.get_equity_curve_for_experiment(experiment=experiment, portfolio_size=k) for k in portfolio_sizes
    }
    returns = pd.concat(curves, axis=1)
    returns.columns = list(portfolio_sizes)
    statistics = evaluation.compute_portfolio_statistics(returns)
    dataset = experiment.get_sample().get_dataset(DatasetType.TEST)
    btc = evaluation.get_btc_buy_and_hold_baseline(dataset)
    btc_statistics = evaluation.compute_btc_buy_and_hold_statistics(btc)
    aligned = evaluation.align_portfolio_returns_with_btc(returns.rename(columns=lambda k: f"K={k}"), btc)
    statistics.insert(1, "cumulative return", returns.sum())
    statistics.loc[btc_statistics.name] = {
        "average event return": float("nan"),
        **btc_statistics.to_dict(),
    }
    print(f"Portfolio model selected on validation: {experiment.get_experiment_name()}")
    print(f"BTC benchmark interval: {btc.attrs['start_time']} to {btc.attrs['end_time']}")
    return aligned, statistics


## Auc bootstrap report

In [ ]:
"""Distribution plots and replayable artifacts for the all-model test AUC bootstrap."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pumps_and_dumps.backtest.robustness.auc_bootstrap import AucBootstrapResults


def plot_test_auc_distributions(result: AucBootstrapResults) -> plt.Figure:
    """Show central bootstrap intervals, medians and test sample point estimates."""
    names = list(result.summary.index)
    fig, interval_ax = plt.subplots(figsize=(11, max(7, len(names) * 0.45 + 2)), layout="constrained")
    colors = plt.get_cmap("tab20")(np.linspace(0, 1, len(names)))
    ticks = []
    for position, (name, color) in enumerate(zip(names, colors)):
        samples = result.bootstrap_aucs[name].to_numpy()
        lower, q1, median, q3, upper = np.quantile(samples, [result.alpha / 2, 0.25, 0.5, 0.75, 1 - result.alpha / 2])
        interval_ax.plot([lower, upper], [position, position], color=color, linewidth=2)
        interval_ax.plot([q1, q3], [position, position], color=color, linewidth=8, solid_capstyle="butt")
        interval_ax.plot(median, position, "|", color="black", markersize=12)
        interval_ax.plot(
            result.summary.loc[name, "point_estimate"], position, "o", color=color, markeredgecolor="black"
        )
        ticks.append(f"M{position + 1}  {name}")
    interval_ax.set_yticks(range(len(names)), ticks)
    interval_ax.invert_yaxis()
    interval_ax.set_title(
        f"Central 50% (thick) and {100 * (1 - result.alpha):g}% (thin)\nDot: test sample point estimate; tick: bootstrap median"
    )
    interval_ax.set_xlabel("Test Top@k%AUC (normalised, k = 0–20%)")
    interval_ax.grid(alpha=0.2)
    fig.suptitle(
        f"{len(result.bootstrap_aucs):,} paired test resamples · {len(result.event_aucs)} pump events · fixed models"
    )
    return fig


def export_test_auc_bootstrap(result: AucBootstrapResults, directory: Path, *, provenance: dict | None = None) -> Path:
    """Export draws, event contributions, predictions, summaries and assumptions."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    result.summary.to_csv(directory / "auc_summary.csv")
    result.pairwise.to_csv(directory / "auc_pairwise.csv", index=False)
    result.bootstrap_aucs.to_csv(directory / "auc_distributions.csv")
    result.event_aucs.to_csv(directory / "event_aucs.csv")
    np.savez_compressed(
        directory / "resampled_events.npz",
        event_ids=result.event_aucs.index.to_numpy(dtype=str),
        sampled_indices=result.sampled_indices,
    )
    predictions = pd.concat(result.scored_datasets, names=["model", "row"]).reset_index()
    predictions.to_parquet(directory / "test_predictions.parquet", index=False)
    metadata = {
        "metric": "normalised Top@k%AUC",
        "auc_bins": result.bins.tolist(),
        "dataset_type": "TEST",
        "n_events": len(result.event_aucs),
        "n_bootstrap": len(result.bootstrap_aucs),
        "models": list(result.event_aucs.columns),
        "random_state": result.random_state,
        "alpha": result.alpha,
        "resampling": "N whole pump events with replacement; identical draws for all fixed models",
        "ci_method": "pointwise percentile; conditional on fitted models; independent events assumed",
        "p_value_method": "two-sided centred paired bootstrap with add-one correction; Holm over all model pairs",
        "interpretation": "Exploratory test comparison; the previously inspected test set is not a new untouched holdout",
        "provenance": provenance or {},
    }
    (directory / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return directory


## Training

In [ ]:
"""Construct named experiments using the existing pipeline training methods."""

from pumps_and_dumps.backtest.pipelines.base_model import TrainingPipeline
from pumps_and_dumps.backtest.experiment import Experiment


def train_experiment(
    pipeline: TrainingPipeline, name: str, tuned: bool = True
) -> Experiment:
    """Train one named experiment without changing pipeline settings or splits."""
    sample = pipeline.create_sample()
    return Experiment(name, pipeline.train(sample=sample, tuned=tuned), sample)


In [ ]:
# Integrity checks and run records stay out of the narrative notebook.
import hashlib
import importlib.metadata
from datetime import datetime, timezone
from research_snapshots.reproducibility import reproducibility_settings
from pumps_and_dumps.core.paths import FEATURE_DIR, DATA_DIR, get_root_dir

def validate_research_inputs(frame, manifest_path, provenance):
    from features.utils import load_pumps
    expected = {str(event) for event in load_pumps(path=manifest_path)}
    if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != provenance["active_pumps_sha256"]:
        raise ValueError("Pump manifest differs from the pinned historical paper cohort")
    if set(frame.pump_hash.unique()) != expected:
        raise ValueError("The dataset does not contain every requested pump event")

def record_research_run(experiments, directory, manifest_path, provenance):
    from research_snapshots.artifacts import save_models
    fitted_models = save_models(experiments, directory)
    records = []
    event_ids = {event for _, dataset in experiments[0].get_sample().iter_datasets()
                 for event in dataset.all_data().pump_hash.unique()}
    for path in [FEATURE_DIR / "pumps" / f"{event}.parquet" for event in sorted(event_ids)]:
        records.append({"file": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    pd.DataFrame(records).to_csv(directory / "feature_hashes.csv", index=False)
    splits = {}
    for kind, dataset in experiments[0].get_sample().iter_datasets():
        data = dataset.all_data()
        splits[kind.name] = {"events": data.pump_hash.nunique(), "rows": len(data),
                            "positive": int(data.is_pumped.sum()), "negative": int((~data.is_pumped).sum()),
                            "average_cross_section": len(data) / data.pump_hash.nunique()}
        data[["pump_hash", "pump_time"]].drop_duplicates().to_csv(directory / f"{kind.name.lower()}_events.csv", index=False)
    metadata = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "fitted_models": fitted_models,
        "reproducibility": reproducibility_settings(),
        "model_parameters": {e.get_experiment_name(): e.get_model().params for e in experiments},
        "feature_order": {e.get_experiment_name(): next(e.get_sample().iter_datasets())[1].feature_set.regressors for e in experiments},
        "pump_manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        "model_code_commit": provenance["commit"], "runtime_sha256": provenance["runtime_sha256"],
        "split_counts": splits, "paper_cohort": provenance.get("paper_cohort"),
        "features": "Existing regenerated feature files supplied by the user; not rebuilt in this run",
        "studies_sha256": hashlib.sha256((DATA_DIR / "studies.db").read_bytes()).hexdigest(),
        "versions": {name: importlib.metadata.version(name) for name in
                     ["numpy", "pandas", "polars", "scikit-learn", "catboost", "optuna", "numba", "statsmodels"]},
        "source_hashes": {str(path.relative_to(get_root_dir())): hashlib.sha256(path.read_bytes()).hexdigest()
                          for path in get_root_dir().glob("notebooks/*.ipynb")},
    }
    (directory / "run_manifest.json").write_text(json.dumps(metadata, indent=2, default=str) + "\n")


### Logistic regression coefficients and explanatory inference

In [ ]:
def logistic_coefficient_report(experiment, baseline_model):
    """Report predictive coefficients alongside the already fitted statsmodels baseline."""
    from backtest.utils.sample import DatasetType as HistoricalDatasetType

    training = experiment.get_sample().get_dataset(HistoricalDatasetType.TRAIN)
    X = training.get_data().astype(float)
    y = training.get_label().astype(int)
    estimator = experiment.get_model()._model
    coefficient = pd.Series(estimator.coef_[0], index=estimator.feature_names_in_, name="coefficient")
    mean, sd = X.mean(), X.std(ddof=1)
    coefficients = coefficient.to_frame().assign(
        odds_ratio=np.exp(coefficient), training_sd=sd,
        coefficient_per_training_sd=coefficient * sd,
        odds_ratio_per_training_sd=np.exp(coefficient * sd),
    )
    coefficients.loc["intercept"] = [estimator.intercept_[0], np.exp(estimator.intercept_[0]), np.nan, np.nan, np.nan]
    coefficients.index.name = "feature"
    result = baseline_model.result
    groups = training.all_data()["pump_hash"]
    diagnostics = {
        "predictive_model": experiment.get_experiment_name(),
        "predictive_parameters": estimator.get_params(),
        "nonzero_predictive_coefficients": int(coefficient.ne(0).sum()),
        "statsmodels_model": "Unweighted, unpenalized statsmodels baseline; separate from the tuned L1 primary model",
        "statsmodels_scaling": "Every preprocessed predictor centered and divided by its training sample SD",
        "covariance": "Pump-event clustered, with small-sample correction; independent events assumed",
        "n_observations": len(X), "n_training_events": int(groups.nunique()),
        "n_predictors": X.shape[1], "design_rank": int(np.linalg.matrix_rank(result.model.exog)),
        "converged": bool(result.mle_retvals["converged"]),
        "iterations": int(result.mle_retvals["iterations"]),
        "log_likelihood": float(result.llf), "pseudo_r_squared": float(result.prsquared),
        "aic": float(result.aic), "bic": float(result.bic),
        "max_absolute_mean_score": float(np.abs(result.model.score(result.params)).max() / len(X)),
    }
    scaling = pd.DataFrame({"training_mean": mean, "training_sd": sd}).rename_axis("feature")
    return coefficients, result, scaling, diagnostics


### Logistic regularization comparison

Fit the L2 candidate on training rows and choose its penalty using validation event AUC only.

In [ ]:
from research_snapshots.reproducibility import RANDOM_SEED

def fit_l2_logistic_path(sample, reference_params, c_values, *, seed=RANDOM_SEED):
    """Select C on validation only; reuse the historical L1 data and class weights."""
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    from backtest.pipelines.LogisticRegression.model import LogisticRegressionModel
    from backtest.utils.metrics import calculate_topk_percent_auc
    from backtest.utils.sample import DatasetType as HistoricalDatasetType

    values = np.unique(np.asarray(c_values, dtype=float))
    if not len(values) or not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("C candidates must be finite and strictly positive")
    params = dict(reference_params) | {"penalty": "l2", "solver": "liblinear", "tol": 1e-8, "max_iter": 2000, "random_state": seed}
    validation = sample.get_dataset(HistoricalDatasetType.VALIDATION)
    records = []
    best_model, best_auc = None, -np.inf
    for c_value in values:
        model = LogisticRegressionModel(params=params | {"C": float(c_value)})
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.train(sample)
        score = calculate_topk_percent_auc(model, validation)
        if not np.isfinite(score):
            raise RuntimeError("Non-finite validation AUC for L2 logistic regression")
        records.append({"C": float(c_value), "validation_topk_percent_auc": score,
                        "iterations": int(model._model.n_iter_.max())})
        # Ascending C and strict improvement break ties toward stronger regularization.
        if score > best_auc:
            best_model, best_auc = model, score
    return best_model, pd.DataFrame(records)


def run_fixed_logistic_subset_robustness(params, **kwargs):
    """Vary training events while preserving the chosen penalty and hyperparameters."""
    from copy import deepcopy
    from backtest.pipelines.LogisticRegression.pipe import LogisticRegressionPipeline
    from backtest.pipelines.LogisticRegression.model import LogisticRegressionModel
    from backtest.robust.robustness import run_cross_section_subset_robustness

    class FixedLogisticPipeline(LogisticRegressionPipeline):
        def train(self, sample, tuned=False):
            return LogisticRegressionModel(params=deepcopy(params)).train(sample)

    return run_cross_section_subset_robustness(pipeline_factory=FixedLogisticPipeline, **kwargs)


### Uniform random portfolio reference

Resample assets within each fixed test event, retaining the eligible universe and the same execution rules. Portfolio metrics describe individual draws; the mean curve and pointwise ranges summarize asset-selection randomness.

In [ ]:
def random_portfolio_asset_returns(dataset, manager=None):
    """Execute each eligible asset once at a fixed per-asset notional; return normalized ROE."""
    from backtest.portfolio.TOPKPortfolio import TOPKPortfolio
    from core.currency_pair import CurrencyPair as HistoricalCurrencyPair
    from core.time_utils import Bounds as HistoricalBounds

    manager = manager or TOPKPortfolio(model=None, use_price_impact=False, order_notional_usdt=1.0)
    notional = float(manager.order_notional_usdt)
    if not np.isfinite(notional) or notional <= 0 or manager.order_notional_quote != 0:
        raise ValueError("Asset execution requires a finite positive USDT notional and no quote-notional override")
    cross_sections = dataset.all_data().groupby("pump_hash", sort=False)
    pumps = sorted(dataset.get_pumps())
    records = []
    for event_index, pump in enumerate(pumps):
        cross_section = cross_sections.get_group(pump.as_pump_hash())
        assets = sorted(cross_section.currency_pair.tolist())
        if not assets or len(set(assets)) != len(assets):
            raise ValueError("Expected one row per eligible asset in each event")
        for asset_index, symbol in enumerate(assets):
            cp = HistoricalCurrencyPair.from_string(symbol)
            is_target = pump.is_manipulated(cp)
            exit_after = pump.time + (manager._sell_after if is_target else timedelta(0))
            decision = pump.time - manager._buy_before
            # The original engine reads two days but uses only the first entry/exit.
            # Extend to its original cutoff for thin assets whose exit arrives later.
            prices = manager.load_price_ts(HistoricalBounds(decision, exit_after + timedelta(minutes=1)), cp)
            has_entry = ((prices.index >= decision) & (prices.index < pump.time)).any()
            if has_entry and not (prices.index >= exit_after).any():
                prices = manager.load_price_ts(HistoricalBounds(decision, pump.time + timedelta(days=1)), cp)
            execute = manager.pumped_transaction if is_target else manager.regular_transaction
            tx = execute(ts_price=prices, pump=pump, cp=cp)
            pnl = 0.0 if tx.is_empty() else manager._pnl_calculator.calculate_transaction_pnl(tx)
            records.append({
                "event_index": event_index, "pump_hash": pump.as_pump_hash(), "pump_time": pump.time,
                "asset_index": asset_index, "currency_pair": symbol, "is_pumped": is_target,
                "executed": not tx.is_empty(), "return": pnl / notional,
            })
            if manager.use_price_impact:
                records[-1].update({
                    "pnl_usdt": pnl, "order_notional_usdt": notional,
                    "entry_ts": tx.entry_ts, "exit_ts": tx.exit_ts,
                    "entry_impact_bps": tx.entry_impact_bps, "exit_impact_bps": tx.exit_impact_bps,
                    "entry_impact_num_bars": tx.entry_impact_num_bars,
                    "exit_impact_num_bars": tx.exit_impact_num_bars,
                })
        if (event_index + 1) % 10 == 0 or event_index + 1 == len(pumps):
            print(f"Random portfolio execution inputs: {event_index + 1}/{len(pumps)} events", flush=True)
    return pd.DataFrame(records)


def resample_random_portfolios(asset_returns, portfolio_sizes=(1, 2, 5, 10, 20, 30), n_resamples=10_000, seed=42):
    """Uniform subsets within fixed events, with shared random orders across k."""
    from backtest.utils.evaluation import compute_portfolio_statistics as portfolio_statistics

    sizes = np.asarray(portfolio_sizes, dtype=int)
    if not len(sizes) or (sizes < 1).any() or len(np.unique(sizes)) != len(sizes) or n_resamples < 2:
        raise ValueError("Use distinct positive portfolio sizes and at least two resamples")
    assets = asset_returns.sort_values(["event_index", "asset_index"]).reset_index(drop=True)
    if assets.empty or not np.isfinite(assets["return"]).all():
        raise ValueError("Every eligible asset needs a finite return, including zero for unfilled legs")
    events = assets.groupby("event_index", sort=True)
    counts = events.size().to_numpy()
    if (counts < sizes.max()).any():
        raise ValueError("Portfolio size exceeds the eligible universe")
    if assets.duplicated(["event_index", "currency_pair"]).any():
        raise ValueError("Duplicate event/asset rows")
    times = pd.DatetimeIndex(events.pump_time.first(), name="time")
    if not times.is_monotonic_increasing:
        raise ValueError("Events must be in chronological order")
    rng = np.random.default_rng(seed)
    indices = np.empty((n_resamples, len(counts), sizes.max()), dtype=np.int32)
    returns = np.empty((n_resamples, len(counts), len(sizes)))
    hits = np.empty_like(returns, dtype=bool)
    for event_index, (_, event) in enumerate(events):
        # Random keys induce a uniform permutation, without replacement within a portfolio.
        order = rng.random((n_resamples, len(event))).argsort(axis=1)[:, :sizes.max()]
        indices[:, event_index] = order
        sums = np.cumsum(event["return"].to_numpy()[order], axis=1)
        returns[:, event_index] = sums[:, sizes - 1] / sizes
        target_counts = np.cumsum(event.is_pumped.to_numpy()[order], axis=1)
        hits[:, event_index] = target_counts[:, sizes - 1] > 0
    metrics = []
    paths = []
    for column, k in enumerate(sizes):
        event_returns = pd.DataFrame(returns[:, :, column].T, index=times)
        stats = portfolio_statistics(event_returns)
        stats.insert(1, "cumulative return", event_returns.sum())
        stats["Top@k hit rate"] = hits[:, :, column].mean(axis=1)
        stats.index.name = "replicate"
        stats = stats.reset_index()
        stats.insert(0, "k", int(k))
        metrics.append(stats)
        # Aggregate coincident events before pointwise quantiles, then retain quiet days.
        daily = event_returns.groupby(event_returns.index.normalize()).sum()
        daily = daily.reindex(pd.date_range(times.min().normalize(), times.max().normalize(), freq="D"), fill_value=0)
        cumulative = daily.cumsum().to_numpy()
        paths.append(pd.DataFrame({
            "time": daily.index, "k": int(k), "mean": cumulative.mean(axis=1),
            "lower": np.quantile(cumulative, 0.025, axis=1),
            "upper": np.quantile(cumulative, 0.975, axis=1),
        }))
    metrics = pd.concat(metrics, ignore_index=True)
    long = metrics.melt(id_vars=["k", "replicate"], var_name="metric", value_name="value")
    summary = long.groupby(["k", "metric"], sort=False).value.agg(
        mean="mean", lower=lambda x: x.quantile(0.025), upper=lambda x: x.quantile(0.975),
    )
    return {
        "asset_returns": assets, "indices": indices, "event_returns": returns,
        "mean_returns": pd.DataFrame(returns.mean(axis=0), index=times, columns=sizes),
        "metrics": metrics, "summary": summary, "paths": pd.concat(paths, ignore_index=True),
        "seed": seed, "portfolio_sizes": sizes.tolist(), "n_resamples": n_resamples,
    }


def compare_random_portfolio_metrics(model_statistics, random_result):
    """Descriptive differences conditional on these test events, not bootstrap p-values."""
    rows = []
    for k, group in random_result["metrics"].groupby("k"):
        for metric in model_statistics.columns:
            model_value = float(model_statistics.loc[k, metric])
            values = group[metric].dropna().to_numpy()
            differences = model_value - values
            rows.append({
                "k": k, "metric": metric, "model": model_value, "random mean": values.mean(),
                "model minus random mean": differences.mean(),
                "difference lower": np.quantile(differences, 0.025),
                "difference upper": np.quantile(differences, 0.975),
                "fraction random >= model": float((values >= model_value).mean()),
            })
    return pd.DataFrame(rows).set_index(["k", "metric"])


def export_random_portfolios(result, directory):
    """Save every selection and per-draw metric for replay without market-data reads."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    result["asset_returns"].to_parquet(directory / "asset_returns.parquet", index=False)
    np.savez_compressed(directory / "resampled_assets.npz", asset_indices=result["indices"],
                        event_returns=result["event_returns"], portfolio_sizes=result["portfolio_sizes"])
    result["metrics"].to_csv(directory / "metrics.csv", index=False)
    result["summary"].to_csv(directory / "summary.csv")
    result["paths"].to_csv(directory / "cumulative_paths.csv", index=False)
    result["mean_returns"].to_csv(directory / "mean_event_returns.csv")
    metadata = {
        "seed": result["seed"], "n_resamples": result["n_resamples"],
        "portfolio_sizes": result["portfolio_sizes"], "rng": "numpy.default_rng / PCG64",
        "selection": "Uniform random keys per eligible asset/event/draw; take first k of sorted keys, without replacement",
        "weights": "1/k; unfilled legs retain zero-return cash, with no renormalization",
        "events": "All test events remain in chronological order; events are not resampled",
        "execution": result.get("execution", "Shared no-impact engine; entry T-15min, ordinary exit T, target exit T+1min; 25bp round-trip cost; USDT cash-flow conversion"),
        "aggregation": "Additive event returns, no reinvestment; zero-filled daily calendar, 365-day annualization, ddof=1 volatility, zero risk-free rate",
        "intervals": "Pointwise 2.5/97.5 percentiles across asset selections; conditional randomization ranges, not event-bootstrap confidence intervals",
        "statistics": "Each draw's return, volatility and Sharpe computed separately before averaging; plotted mean path is an expectation, not one traded portfolio",
        "replay": "asset_indices[draw,event,position] indexes asset_returns sorted by event_index then asset_index within each event; event_returns[draw,event,k_index]",
        "asset_returns_sha256": hashlib.sha256((directory / "asset_returns.parquet").read_bytes()).hexdigest(),
    }
    (directory / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")


In [ ]:
# Notebook presentation: expected missing-fill notices do not change execution.
import logging


class _NotebookMissingDataFilter(logging.Filter):
    def filter(self, record):
        expected = record.getMessage().startswith(("No data to get prices for ", "No data present for "))
        return record.levelno >= logging.ERROR or not expected


def silence_missing_data_logs():
    root_logger = logging.getLogger()
    if not any(item.name == "notebook_missing_data" for item in root_logger.filters):
        root_logger.addFilter(_NotebookMissingDataFilter("notebook_missing_data"))
